# EDA — Real Estate Pricing

Dataset: [Jiff's House Price Prediction](https://www.kaggle.com/datasets/elakiricoder/jiffs-house-price-prediction-dataset)  
Modelo: Random Forest Classifier (faixa de preço)

---

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from src.data_loader import load_and_prepare
from src.config import TARGET_COL, PRICE_LABELS

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.2f}'.format)

df = load_and_prepare()
print(f'Shape: {df.shape}')
df.head()

## 1. Visão Geral

In [ ]:
df.info()

In [ ]:
df.describe().T.round(2)

In [ ]:
# Valores ausentes
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

## 2. Distribuição de Preços

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df[TARGET_COL], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição de Preços')
axes[0].set_xlabel('Preço (USD)')
axes[0].set_ylabel('Frequência')

axes[1].hist(np.log1p(df[TARGET_COL]), bins=60, color='darkorange', edgecolor='white')
axes[1].set_title('Distribuição log(Preço + 1)')
axes[1].set_xlabel('log(Preço + 1)')

plt.tight_layout()
plt.show()

In [ ]:
# Faixas de preço
ax = df['price_category'].value_counts().reindex(PRICE_LABELS).plot(
    kind='bar', color=['#2ecc71','#3498db','#9b59b6','#e67e22','#e74c3c'],
    figsize=(9, 4), edgecolor='white', rot=0
)
ax.set_title('Imóveis por Faixa de Preço')
ax.set_xlabel('Faixa')
ax.set_ylabel('Contagem')
plt.tight_layout()
plt.show()

## 3. Correlação com o Preço

In [ ]:
num_df = df.select_dtypes(include=[np.number])
corr_price = num_df.corr()[TARGET_COL].drop(TARGET_COL).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
corr_price.plot(kind='barh', color=corr_price.apply(lambda v: '#2ecc71' if v > 0 else '#e74c3c'))
plt.title('Correlação das features com o preço')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

corr_price

## 4. Mapa de Calor — Correlação Completa

In [ ]:
plt.figure(figsize=(14, 10))
corr_matrix = num_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', vmin=-1, vmax=1, linewidths=0.5
)
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()

## 5. Features vs. Preço

In [ ]:
top_features = corr_price.abs().head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

sample = df.sample(min(2000, len(df)), random_state=42)

for i, feat in enumerate(top_features):
    axes[i].scatter(
        sample[feat], sample[TARGET_COL],
        alpha=0.3, s=15, color='steelblue'
    )
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Preço (USD)')
    axes[i].set_title(f'{feat} vs. price')

plt.tight_layout()
plt.show()

## 6. Box Plots por Faixa de Preço

In [ ]:
cat_order = [c for c in PRICE_LABELS if c in df['price_category'].cat.categories]

for feat in ['bedrooms', 'bathrooms', 'house_size', 'grade']:
    if feat not in df.columns:
        continue
    plt.figure(figsize=(9, 4))
    df_plot = df[df['price_category'].isin(cat_order)]
    sns.boxplot(
        data=df_plot, x='price_category', y=feat,
        order=cat_order, palette='Set2'
    )
    plt.title(f'{feat} por faixa de preço')
    plt.tight_layout()
    plt.show()

## 7. Treino Rápido + Feature Importances

In [ ]:
from src.model import train, get_feature_importances

result = train(df, mode='classifier', save=False)
print(f"Acurácia no teste: {result['metrics']['accuracy']:.4f}")

In [ ]:
importances = get_feature_importances(result['pipeline'])
top = importances.head(15)

plt.figure(figsize=(9, 6))
top.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top-15 Features Mais Importantes')
plt.xlabel('Importância')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    result['y_test'],
    result['y_pred'],
    labels=PRICE_LABELS,
    zero_division=0
))